# Preprocessing Dataset Transactions

Dataset: `data/Transactions.csv` (transaksi 3 outlet, Jan-Sep 2025).

## 1. Muat Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = '../data/Transactions.csv'
df = pd.read_csv(CSV)
df.head()

C:\Users\Radit\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Radit\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0


## 2. Cek Kualitas Data

In [2]:
print('Baris          :', df.shape[0])
print('Kolom          :', df.shape[1])
print('Missing value  :', df.isna().sum().sum())
print('Struk duplikat :', df['ReceiptNumber'].duplicated().sum())

Baris          : 53820
Kolom          : 11
Missing value  : 0
Struk duplikat : 0


## 3. Pembersihan

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
df['hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
df['is_weekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)
df['Items'] = df['Items'].str.replace('Houesblend', 'Houseblend', regex=False)
df['is_free'] = (df['TotalAmount'] == 0).astype(int)
df = df.drop_duplicates(subset='ReceiptNumber')
print(df.shape)
df.head()

(53820, 14)


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard,hour,is_weekend,is_free
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0,7,0,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0,7,0,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,7,0,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0,7,0,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0,7,0,0


## 4. Fitur Turunan

In [4]:
fitur = df[['Outlet', 'TotalItem', 'TotalAmount', 'PaymentMethod', 'hour', 'is_weekend', 'is_free']].copy()
fitur.head()

,Outlet,TotalItem,TotalAmount,PaymentMethod,hour,is_weekend,is_free
0,SHOP001,1,23000,BCA,7,0,0
1,SHOP001,1,20000,BCA,7,0,0
2,SHOP002,2,44000,Cash,7,0,0
3,SHOP002,2,40000,Cash,7,0,0
4,SHOP002,2,40000,Cash,7,0,0


## 5. Encoding & Scaling

In [5]:
from sklearn.preprocessing import StandardScaler

X = pd.get_dummies(fitur, columns=['Outlet', 'PaymentMethod'], drop_first=True)
num_cols = ['TotalItem', 'TotalAmount', 'hour']

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
print(X.shape)
X.head()

(53820, 9)


,TotalItem,TotalAmount,hour,is_weekend,is_free,Outlet_SHOP002,Outlet_SHOP003,PaymentMethod_Cash,PaymentMethod_GoFood
0,-0.422530,-0.127397,-1.788894,0,0,False,False,False,False
1,-0.422530,-0.393098,-1.788894,0,0,False,False,False,False
2,2.366696,1.732510,-1.788894,0,0,True,False,True,False
3,2.366696,1.378242,-1.788894,0,0,True,False,True,False
4,2.366696,1.378242,-1.788894,0,0,True,False,True,False
